In [ ]:
"""This is an inv management problem. Finds best policy for the lowest cost.
On every iteration, improve priority_v1 over the priority_vX methods from previous iterations.
Make only small changes.
Try to make the code short.
"""
from scipy.optimize import minimize
import or_gym
import numpy as np
import funsearch
@funsearch.run
def evaluate(n) -> int:
  results = solve()
  max_value = sum(results)/len(results)
  return int(max_value)




def solve():
  # Register environment
  def dfo_func(policy, env, *args):
    '''
    Runs an episode based on current base-stock model 
    settings. This allows us to use our environment for the 
    DFO optimizer.
    '''
    env.reset() # Ensure env is fresh
    rewards = []
    done = False
    while not done:
        action = priority(policy, env)
        state, reward, done, _ = env.step(action)
        rewards.append(reward)
        if done:
            break
            
    rewards = np.array(rewards)
    prob = env.demand_dist.pmf(env.D, **env.dist_param)
    
    # Return negative of expected profit
    return -1 / env.num_periods * np.sum(prob * rewards)
  
  def optimize_inventory_policy(env_name, fun,
    init_policy=None, env_config={}, method='Powell'):
    
    env = or_gym.make(env_name, env_config=env_config)
    
    if init_policy is None:
        init_policy = np.ones(env.num_stages-1)
        
    # Optimize policy
    out = minimize(fun=fun, x0=init_policy, args=env, 
        method=method)
    
    policy = out.x.copy()
    
    # Policy must be positive integer
    policy = np.round(np.maximum(policy, 0), 0).astype(int)
    
    return policy, out
  env_name='InvManagement-v1'
  env_config = {}
  policy, out = optimize_inventory_policy('InvManagement-v1',
    dfo_func,init_policy)
  print("Re-order levels: {}".format(policy))
  print("DFO Info:\n{}".format(out))

  env = or_gym.make(env_name, env_config=env_config)
  eps = 1000
  rewards = []
  for i in range(eps):
      env.reset()
      reward = 0
      while True:
          action = priority(policy, env)
          s, r, done, _ = env.step(action)
          reward += r
          if done:
              rewards.append(reward)
              break
  return rewards


In [ ]:

@funsearch.evolve
def priority(policy, env):   
    if policy is None:
        return np.ones(env.num_stages - 1) * 3  # Using 3 parameters per stage

    # Initialize state variables
    params = policy.reshape(-1, 3)  # [s, S, alpha] for each stage
    s, S, alpha = params.T

    # Calculate echelon inventory levels
    if env.period == 0:
        inv_ech = np.cumsum(env.I[env.period] + env.T[env.period])
    else:
        inv_ech = np.cumsum(env.I[env.period] + env.T[env.period] - env.B[env.period - 1, :-1])

    # Introduce dynamic demand adjustment
    demand_forecast = alpha * env.D[env.period] + (1 - alpha) * inv_ech

    # Unconstrained actions considering dynamic demand adjustment
    unc_actions = np.where(inv_ech < s, S - demand_forecast, 0)

    # Ensure actions respect constraints
    inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
    actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))
    
    return actions


In [252]:

@funsearch.evolve
def priority(policy, env):  
  """Further improved version of `priority_v1`."""
  if policy is None:
      return np.ones(env.num_stages * 2 + 1)

  s = policy[:env.num_stages-1]
  S = policy[env.num_stages-1:2*(env.num_stages-1)]
  reorder_point_multiplier = policy[-1]

  if env.period == 0:
      inv_ech = np.cumsum(env.I[env.period] + env.T[env.period])
  else:
      inv_ech = np.cumsum(env.I[env.period] + env.T[env.period] - env.B[env.period - 1, :-1])

  demand_forecast = np.mean(env.D[max(env.period-3, 0):env.period+1]) 
  variability_factor = np.std(env.D[max(env.period-3, 0):env.period+1]) 
  safety_stock_factor = np.tanh(variability_factor)
  
  reorder_point = s + demand_forecast * safety_stock_factor * reorder_point_multiplier  

  unc_actions = np.where(inv_ech < reorder_point, S - inv_ech, 0)

  inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
  actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))

  return actions


In [ ]:
import random
def generate_random_dist_params(dist):
    '''
    Generates random parameters for the customer demand distribution based on the dist value.
    '''
    dist_params = {}
    if dist == 1:  # Poisson distribution
        dist_params['mu'] = random.randint(10, 50)
    elif dist == 2:  # Binomial distribution mean np and variance np(1-p)
        dist_params['n'] = random.randint(20, 100)
        dist_params['p'] = round(random.uniform(0.1, 0.9), 2)
    elif dist == 3:  # Uniform random integer distribution mean (low+high)/2, variance (high-low+1)^2/12
        dist_params['low'] = random.randint(1, 10)
        dist_params['high'] = random.randint(20, 50)
    elif dist == 4:  # Geometric distribution mean 1/p, variance (1-p)/p^2
        dist_params['p'] = round(random.uniform(0.1, 0.9), 2)
    
    return dist_params
def generate_env_configs(num_configs=10):
    '''
    Generates a list of environment configurations with random parameters,
    where the length of lists is the same as the input lengths (hardcoded).
    
    Parameters:
    num_configs: int
        The number of different environment configurations to generate.
        
    Returns:
    List of dictionaries, each containing a set of parameters for the simulation.
    '''
    configs = []
    periods = random.randint(30, 200)#fix the number of periods
    for _ in range(num_configs):
        dis=random.randint(1,5)
        config = {
            'periods': periods,  # Keep the number of periods the same as input
            'I0': [random.randint(50, 200) for _ in range(3)],  # Same length as input I0
            'p': round(random.uniform(1.5, 3.0), 2),  # Random unit price between 1.5 and 3.0
            'r': [round(random.uniform(0.5, 2.0), 2) for _ in range(4)],  # Same length as input r
            'k': [round(random.uniform(0.02, 0.15), 3) for _ in range(4)],  # Same length as input k
            'h': [round(random.uniform(0.05, 0.2), 2) for _ in range(3)],  # Same length as input h
            'c': [random.randint(50, 150) for _ in range(3)],  # Same length as input c
            'L': [random.randint(1, 15) for _ in range(3)],  # Same length as input L
            #'backlog': random.choice([True, False]),  # Randomly choose if unfulfilled orders are backlogged
            'dist': dis,  # Randomly choose distribution type(there are bugs in the code if we change this)
            'dist_param': generate_random_dist_params(dis),  # Generate random distribution parameters
            'alpha': round(random.uniform(0.9, 1.0), 2),  # Random discount factor between 0.9 and 1.0
            'seed_int': random.randint(0, 100),  # Random seed for the random state
            'user_D': np.random.randint(0, 50, size=periods),  # User-supplied demand same length as periods
            '_max_rewards': random.randint(1000, 3000)  # Random maximum rewards
        }
        configs.append(config)
    
    return configs

#how to create more test cases:
#use only different distributions? use different parameters for the distributions?
#or use completely different structures for the environment?
#my thoughts is to dont change the unit price
#rn, the score is negtive if we use the policy found by the optimizer in one env and polutate it to other environments
#also, i need the optimal solution for comparison (maybe using solver to get one)

In [270]:
import random
def generate_random_dist_params(dist):
    '''
    Generates random parameters for the customer demand distribution based on the dist value.
    '''
    dist_params = {}
    if dist == 1:  # Poisson distribution
        dist_params['mu'] = 15
    elif dist == 2:  # Binomial distribution mean 15 and variance 7.5
        dist_params['n'] = 30
        dist_params['p'] = 0.5
    elif dist == 3:  # Uniform random integer distribution mean 16, variance 100
        dist_params['low'] = 1
        dist_params['high'] = 31
    elif dist == 4:  # Geometric distribution
        dist_params['p'] = 1/15
    
    return dist_params
def generate_env_configs(num_configs=10):
    '''
    Generates a list of environment configurations with random parameters,
    where the length of lists is the same as the input lengths (hardcoded).
    
    Parameters:
    num_configs: int
        The number of different environment configurations to generate.
        
    Returns:
    List of dictionaries, each containing a set of parameters for the simulation.
    '''
    configs = []
    periods = 30 #fix the number of periods
    for _ in range(num_configs):
        dis=1
        config = {
            'periods': periods,  # Keep the number of periods the same as input
            'I0': [random.randint(150, 200) for _ in range(3)],  # Same length as input I0
            'p': 2,  # Random unit price between 1.5 and 3.0
            'r': [1.5, 1.0, 0.75, 0.5],  # Same length as input r
            'k': [0.10, 0.075, 0.05, 0.025],  # Same length as input k
            'h': [0.15, 0.10, 0.05],  # Same length as input h
            'c': [100, 90, 80],  # Same length as input c
            'L': [3, 5, 10],  # Same length as input L
            #'backlog': random.choice([True, False]),  # Randomly choose if unfulfilled orders are backlogged
            'dist': dis,  # Randomly choose distribution type(there are bugs in the code if we change this)
            'dist_param': generate_random_dist_params(dis),  # Generate random distribution parameters
            'alpha': round(random.uniform(0.9, 1.0), 2),  # Random discount factor between 0.9 and 1.0
            'seed_int': random.randint(0, 100),  # Random seed for the random state
            'user_D': np.random.randint(0, 50, size=periods),  # User-supplied demand same length as periods
            '_max_rewards': 2000 
        }
        configs.append(config)
    
    return configs

#how to create more test cases:
#use only different distributions? use different parameters for the distributions?
#or use completely different structures for the environment?
#my thoughts is to dont change the unit price
#rn, the score is negtive if we use the policy found by the optimizer in one env and polutate it to other environments
#also, i need the optimal solution for comparison (maybe using solver to get one)

In [271]:
new_inv= generate_env_configs(100)

In [269]:
new_inv

[{'periods': 30,
  'I0': [200, 178, 159],
  'p': 2,
  'r': [1.5, 1.0, 0.75, 0.5],
  'k': [0.1, 0.075, 0.05, 0.025],
  'h': [0.15, 0.1, 0.05],
  'c': [100, 90, 80],
  'L': [3, 5, 10],
  'dist': 5,
  'dist_param': {},
  'alpha': 0.91,
  'seed_int': 90,
  'user_D': array([ 9, 39,  0,  1,  3, 14, 40, 15, 48, 47, 30, 36, 24, 35, 23, 20,  3,
         25, 19, 34, 32, 46, 10, 22,  3, 23,  9, 48, 23, 33]),
  '_max_rewards': 2000},
 {'periods': 30,
  'I0': [179, 150, 195],
  'p': 2,
  'r': [1.5, 1.0, 0.75, 0.5],
  'k': [0.1, 0.075, 0.05, 0.025],
  'h': [0.15, 0.1, 0.05],
  'c': [100, 90, 80],
  'L': [3, 5, 10],
  'dist': 3,
  'dist_param': {'low': 1, 'high': 31},
  'alpha': 0.9,
  'seed_int': 8,
  'user_D': array([ 3, 30, 21, 40,  6, 26,  7, 31, 29, 16, 14, 28,  5, 17, 25,  6, 36,
         24, 35, 28, 44, 18, 15, 12, 45, 18, 15, 25,  3,  0]),
  '_max_rewards': 2000},
 {'periods': 30,
  'I0': [189, 187, 178],
  'p': 2,
  'r': [1.5, 1.0, 0.75, 0.5],
  'k': [0.1, 0.075, 0.05, 0.025],
  'h': [0.15, 

In [ ]:

def priority_ss(policy, env):
  '''
  This is a re-order up-to policy for you to start. This means that for
  each node in the network, if the inventory at that node 
  falls below the level denoted by the policy, we will 
  re-order inventory to bring it to the policy level.
  
  Design new policy that fits the problem.and improve the score.
  '''
  #if policy is None, return the shape of the policy  
  #you should use this to first specify the shape of the policy in you implementation base on the env
  if policy is None :
    return np.ones((env.num_stages-1)*2)
  #split the policy into two parts to get s,S, first half is s, second half is S
  s = policy[:len(policy)//2]
  S = policy[len(policy)//2:]
  # Get echelon inventory levels
  if env.period == 0:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period])
  else:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period] - env.B[env.period-1, :-1])
  # Get unconstrained actions
  #for any inventory level below s, order up to S
  unc_actions = np.where(inv_ech < s, S-inv_ech, 0)
  # unc_actions = policy - inv_ech
  # unc_actions = np.where(unc_actions>0, unc_actions, 0)
  # Ensure that actions can be fulfilled by checking 
  # constraints
  inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
  actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))
  return actions



In [ ]:
new_inv

In [160]:
"""This is an inv management problem. Finds best policy for the lowest cost.
On every iteration, improve priority_v1 over the priority_vX methods from previous iterations.
Make only small changes.
Try to make the code short.
"""
from scipy.optimize import minimize
import or_gym
import numpy as np
import funsearch
@funsearch.run
def evaluate(n,priority) -> int:
  results = solve(n,priority)
  max_value = sum(results)/len(results)
  print(max_value)
  return int(max_value)




def solve(n,priority ) :
  # Register environment
  def dfo_func(policy, env, *args):
    '''
    Runs an episode based on current base-stock model 
    settings. This allows us to use our environment for the 
    DFO optimizer.
    '''
    env.reset() # Ensure env is fresh
    rewards = []
    done = False
    while not done:
        action = priority(policy, env)
        state, reward, done, _ = env.step(action)
        rewards.append(reward)
        if done:
            break
            
    rewards = np.array(rewards)

    
    # Return negative of expected profit
    return -1 / env.num_periods * np.sum(rewards)
  
  def optimize_inventory_policy(env_name, fun,
    init_policy=None, env_config={}, method='Powell'):
    
    env = or_gym.make(env_name, env_config=env_config)
    
    if init_policy is None:
        init_policy = np.ones((env.num_stages-1)*2)
        
    # Optimize policy
    out = minimize(fun=fun, x0=init_policy, args=env, 
        method=method)
    
    policy = out.x.copy()
    
    # Policy must be positive integer
    policy = np.round(np.maximum(policy, 0), 0).astype(int)
    
    return policy, out
 
  
  policy_all=[]
  env_name='InvManagement-v1'
  for i in range(n):
    env_config=new_inv[i]
    policy, out = optimize_inventory_policy('InvManagement-v1',
    dfo_func,init_policy=priority(None,or_gym.make(env_name, env_config={})),env_config=env_config)
    policy_all.append(policy)
    
  policy=np.mean(policy_all,axis=0)
  print("Re-order levels: {}".format(policy))

  
  env_config =new_inv[n]
  env = or_gym.make(env_name, env_config=env_config)
  eps = 1000
  rewards = []
  for i in range(eps):
      env.reset()
      reward = 0
      while True:
          action = priority(policy, env)
          s, r, done, _ = env.step(action)
          reward += r
          if done:
              rewards.append(reward)
              break
  return rewards


In [268]:
elva=evaluate(1,priority_old)

Re-order levels: [ 161. 2285.    0.]
-19.98530655560291


In [267]:
elva=evaluate(1,priority_ss)

Re-order levels: [1.00e+00 1.00e+00 1.00e+00 2.05e+02 4.46e+03 1.00e+00]
-72.77961539540624


In [265]:
elva=evaluate(1,priority)

Re-order levels: [ 135.  106. 2485.]
-17.021553612471564


In [ ]:
elva=evaluate(99,priority_old1 )

Re-order levels: [129.88888889  64.55555556 612.88888889]
318.14210000000026


In [272]:
elva=evaluate(99,priority )

Re-order levels: [49.43434343 57.3030303  50.87878788]
-21.50708286762844


In [273]:
elva=evaluate(99,priority_old )

Re-order levels: [58.68686869 10.58585859 12.16161616]
-21.536278231250268


In [274]:
elva=evaluate(99,priority_ss )

Re-order levels: [ 28.83838384  47.45454545  66.19191919 138.85858586  54.56565657
  32.19191919]
-77.40179271157179


In [161]:
@funsearch.evolve
def priority_old(policy, env):
  '''
  This is a re-order up-to policy for you to start. This means that for
  each node in the network, if the inventory at that node 
  falls below the level denoted by the policy, we will 
  re-order inventory to bring it to the policy level.
  
  Design new policy that fits the problem.and improve the score.
  '''
  if policy is None:
    return np.ones(3)
  # Get echelon inventory levels
  if env.period == 0:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period])
  else:
    inv_ech = np.cumsum(env.I[env.period] +
      env.T[env.period] - env.B[env.period-1, :-1])
  # Get unconstrained actions
  unc_actions = policy - inv_ech
  unc_actions = np.where(unc_actions>0, unc_actions, 0)
  # Ensure that actions can be fulfilled by checking 
  # constraints
  inv_const = np.hstack([env.I[env.period, 1:], np.Inf])
  actions = np.minimum(env.c, np.minimum(unc_actions, inv_const))
  return actions



In [63]:
import statistics as stat
print(stat.mean([evaluate(i,priority) for i in range(10)]))

Re-order levels: [24 22  6]
DFO Info:
 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: -1.0699745321313554
       x: [ 2.420e+01  2.151e+01  6.158e+00]
     nit: 4
   direc: [[ 0.000e+00  0.000e+00  1.000e+00]
           [ 2.102e+00 -2.361e-01  3.781e-01]
           [ 8.903e+00  1.503e+01  2.592e+00]]
    nfev: 161
Re-order levels: [24 22  6]
DFO Info:
 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: -1.0699745321313554
       x: [ 2.420e+01  2.151e+01  6.158e+00]
     nit: 4
   direc: [[ 0.000e+00  0.000e+00  1.000e+00]
           [ 2.102e+00 -2.361e-01  3.781e-01]
           [ 8.903e+00  1.503e+01  2.592e+00]]
    nfev: 161
Re-order levels: [24 22  6]
DFO Info:
 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: -1.0699745321313554
       x: [ 2.420e+01  2.151e+01  6.158e+00]
     nit: 4
   direc: [[ 0.000e+00  0.000e+00  1.000e+00]
           [ 2.102e+00 -2.361e-01  3.781e-01]
  